In [1]:
! pip install langchain langchain-groq langchain-neo4j langgraph neo4j python-dotenv

  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached ormsgpack-1.12.2-cp314-cp314-win_amd64.whl.metadata (3.3 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached zstandard-0.25.0-cp314-cp314-win_amd64.whl.metadata (3.3 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/570.0 kB ? eta -:--:--
   ---------------------------------------- 570.0/570.0 kB 4.3 MB/s  0:00:00
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl (41 kB)
   ---------------------------------------- 0.0/744.6 kB ? eta -:--:--
   -


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Connect to Neo4j

In [11]:
import os
import re
import json

from dotenv import load_dotenv
from neo4j import GraphDatabase

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


In [12]:
load_dotenv(override=True)

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")


# Physical Neo4j database
NEO4J_DATABASE = "neo4j"

# Logical graph inside neo4j database
GRAPH_NAME = "university_graph"

In [13]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(
        NEO4J_USERNAME,
        NEO4J_PASSWORD,
    ),
)

driver.verify_connectivity()

print("Connected to Neo4j")
print("Physical database:", NEO4J_DATABASE)
print("Logical graph:", GRAPH_NAME)

Connected to Neo4j
Physical database: neo4j
Logical graph: university_graph


# Verify my graph is exist or not 

In [14]:
with driver.session(
    database=NEO4J_DATABASE
) as session:

    result = session.run(
        """
        MATCH (n {graph_name: $graph_name})
        RETURN count(n) AS total_nodes
        """,
        graph_name=GRAPH_NAME,
    )

    total_nodes = result.single()[
        "total_nodes"
    ]

print(
    "Nodes in logical graph:",
    total_nodes
)

Nodes in logical graph: 7


# Connect to Groq

In [15]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY,
)

print("Groq LLM ready")

Groq LLM ready


# Test Groq

In [17]:
response = llm.invoke(
    "Reply with exactly: Groq connected"
)

print(response.content)

Groq connected


# Read Schema

In [18]:
def get_graph_schema(
    driver,
    database,
    graph_name
):
    schema = {
        "nodes": {},
        "relationships": [],
    }

    with driver.session(
        database=database
    ) as session:

        # -------------------------
        # Node labels + properties
        # -------------------------

        node_result = session.run(
            """
            MATCH (
                n {
                    graph_name: $graph_name
                }
            )

            UNWIND labels(n) AS label
            UNWIND keys(n) AS property

            RETURN
                label,
                collect(
                    DISTINCT property
                ) AS properties

            ORDER BY label
            """,
            graph_name=graph_name,
        )

        for record in node_result:

            schema["nodes"][
                record["label"]
            ] = record["properties"]


        # -------------------------
        # Relationships
        # -------------------------

        relationship_result = session.run(
            """
            MATCH
                (
                    a {
                        graph_name: $graph_name
                    }
                )
                -[r]->
                (
                    b {
                        graph_name: $graph_name
                    }
                )

            RETURN DISTINCT
                labels(a) AS source,
                type(r) AS relationship,
                labels(b) AS target

            ORDER BY
                source,
                relationship,
                target
            """,
            graph_name=graph_name,
        )

        for record in relationship_result:

            schema[
                "relationships"
            ].append(
                {
                    "source":
                        record["source"],

                    "relationship":
                        record[
                            "relationship"
                        ],

                    "target":
                        record["target"],
                }
            )

    return schema

# Read Schema

In [19]:
neo4j_schema = get_graph_schema(
    driver=driver,
    database=NEO4J_DATABASE,
    graph_name=GRAPH_NAME,
)

neo4j_schema

{'nodes': {'Course': ['graph_name',
   'docling_id',
   'identifier',
   'enrolled_students',
   'technologies'],
  'Department': ['graph_name', 'docling_id', 'identifier', 'professors'],
  'Professor': ['graph_name', 'docling_id', 'identifier', 'research_areas'],
  'Student': ['docling_id', 'skills', 'identifier', 'graph_name'],
  'University': ['docling_id', 'identifier', 'graph_name']},
 'relationships': [{'source': ['Department'],
   'relationship': 'HAS_HEAD',
   'target': ['Professor']},
  {'source': ['Department'],
   'relationship': 'HAS_STUDENT',
   'target': ['Student']},
  {'source': ['University'],
   'relationship': 'HAS_DEPARTMENT',
   'target': ['Department']},
  {'source': ['University'],
   'relationship': 'OFFERS_COURSE',
   'target': ['Course']}]}

# Convert Schema to text for LLM

In [20]:
def build_schema_text(
    schema
):
    lines = []

    lines.append(
        "NODE LABELS AND PROPERTIES"
    )

    for label, properties in (
        schema["nodes"].items()
    ):

        lines.append(
            f"\nNode: {label}"
        )

        lines.append(
            "Properties: "
            + ", ".join(properties)
        )


    lines.append(
        "\nRELATIONSHIPS"
    )

    for relation in (
        schema["relationships"]
    ):

        source = ", ".join(
            relation["source"]
        )

        target = ", ".join(
            relation["target"]
        )

        relationship = (
            relation["relationship"]
        )

        lines.append(
            f"({source})"
            f"-[:{relationship}]->"
            f"({target})"
        )

    return "\n".join(lines)

# Build Dynamic Schema

In [21]:
GRAPH_SCHEMA = build_schema_text(
    neo4j_schema
)

print(GRAPH_SCHEMA)

NODE LABELS AND PROPERTIES

Node: Course
Properties: graph_name, docling_id, identifier, enrolled_students, technologies

Node: Department
Properties: graph_name, docling_id, identifier, professors

Node: Professor
Properties: graph_name, docling_id, identifier, research_areas

Node: Student
Properties: docling_id, skills, identifier, graph_name

Node: University
Properties: docling_id, identifier, graph_name

RELATIONSHIPS
(Department)-[:HAS_HEAD]->(Professor)
(Department)-[:HAS_STUDENT]->(Student)
(University)-[:HAS_DEPARTMENT]->(Department)
(University)-[:OFFERS_COURSE]->(Course)


# Prompt for LLM for NL -> Cypher

In [22]:
cypher_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert Neo4j Cypher query generator.

Your job is to convert a natural-language
question into exactly ONE read-only Cypher query.

You MUST follow these rules:

1. Use ONLY node labels shown in the schema.

2. Use ONLY relationship types shown
   in the schema.

3. Use ONLY properties shown in the schema.

4. Never invent schema elements.

5. The Neo4j database contains multiple
   logical graphs.

6. Every node used in the query must belong to:

   graph_name = '{graph_name}'

7. Generate READ-ONLY queries.

Allowed examples:

MATCH
OPTIONAL MATCH
WHERE
WITH
RETURN
UNWIND
ORDER BY
SKIP
LIMIT

Forbidden:

CREATE
MERGE
DELETE
DETACH DELETE
SET
REMOVE
DROP
LOAD CSV

8. Return ONLY the Cypher query.

9. Do not use markdown fences.

10. Do not explain the query.
"""
        ),
        (
            "human",
            """
GRAPH SCHEMA:

{graph_schema}

USER QUESTION:

{question}
"""
        ),
    ]
)

# Build NL to Cypher chain

In [23]:
nl_to_cypher_chain = (
    cypher_prompt
    |
    llm
)

# Function to generate Cypher

In [24]:
def generate_cypher(
    question
):

    response = (
        nl_to_cypher_chain.invoke(
            {
                "question":
                    question,

                "graph_schema":
                    GRAPH_SCHEMA,

                "graph_name":
                    GRAPH_NAME,
            }
        )
    )

    cypher = (
        response
        .content
        .strip()
    )

    # Remove markdown fences
    # if model adds them accidentally

    cypher = re.sub(
        r"^```(?:cypher)?",
        "",
        cypher,
        flags=re.IGNORECASE,
    )

    cypher = re.sub(
        r"```$",
        "",
        cypher,
    )

    return cypher.strip()

# Test NL -> Cypher only

In [25]:
question = (
    "Which students belong "
    "to each department?"
)

cypher_query = generate_cypher(
    question
)

print(cypher_query)

MATCH (d:Department {graph_name:'university_graph'})-[:HAS_STUDENT]->(s:Student {graph_name:'university_graph'})
RETURN d.identifier AS department, s.identifier AS student
ORDER BY d.identifier, s.identifier


In [26]:
FORBIDDEN_CYPHER_KEYWORDS = [
    "CREATE",
    "MERGE",
    "DELETE",
    "DETACH",
    "SET",
    "REMOVE",
    "DROP",
    "LOAD CSV",
    "FOREACH",
]

In [27]:
def validate_read_only_cypher(
    cypher
):

    normalized = (
        cypher
        .upper()
        .strip()
    )

    for keyword in (
        FORBIDDEN_CYPHER_KEYWORDS
    ):

        pattern = (
            rf"\b"
            rf"{re.escape(keyword)}"
            rf"\b"
        )

        if re.search(
            pattern,
            normalized
        ):

            raise ValueError(
                "Unsafe Cypher detected. "
                f"Forbidden operation: "
                f"{keyword}"
            )

    return True

In [28]:
validate_read_only_cypher(
    cypher_query
)

True

# Execute Cypher

In [29]:
def execute_cypher(
    cypher
):

    validate_read_only_cypher(
        cypher
    )

    with driver.session(
        database=NEO4J_DATABASE
    ) as session:

        result = session.run(
            cypher
        )

        records = [
            record.data()
            for record in result
        ]

    return records

# Test Cypher Execution

In [30]:
query_result = execute_cypher(
    cypher_query
)

query_result

[{'department': 'dept_cs', 'student': 'student_001'},
 {'department': 'dept_cs', 'student': 'student_002'}]

# Prompt for Neo4j -> Natural Langugage

In [31]:
answer_prompt = (
    ChatPromptTemplate
    .from_messages(
        [
            (
                "system",
                """
                You answer questions using only
                the supplied Neo4j database result.

                Rules:

                1. Do not invent information.

                2. Do not use information outside
                the database result.

                3. If the result is empty, say
                that no matching data was found.

                4. Answer clearly in natural language.

                5. Do not expose implementation
                details unless the user asks.
                """
                            ),
                            (
                                "human",
                                """
                USER QUESTION:

                {question}

                NEO4J RESULT:

                {query_result}

                Provide the answer.
                """
            ),
        ]
    )
)

# Build Answer Chain

In [32]:
result_to_answer_chain = (
    answer_prompt
    |
    llm
)

# Convert result to Natural Langugage

In [33]:
def generate_answer(
    question,
    query_result
):

    response = (
        result_to_answer_chain
        .invoke(
            {
                "question":
                    question,

                "query_result":
                    json.dumps(
                        query_result,
                        default=str,
                        indent=2,
                    ),
            }
        )
    )

    return (
        response
        .content
        .strip()
    )

In [34]:
answer = generate_answer(
    question=question,
    query_result=query_result,
)

print(answer)

**Department: dept_cs**  
- student_001  
- student_002


# Build NL2Cypher Function

In [35]:
def ask_graph(
    question,
    show_cypher=True,
    show_raw_result=False,
):

    # -------------------------
    # Step 1
    # Natural Language -> Cypher
    # -------------------------

    cypher = generate_cypher(
        question
    )


    # -------------------------
    # Step 2
    # Validate Cypher
    # -------------------------

    validate_read_only_cypher(
        cypher
    )


    # -------------------------
    # Step 3
    # Execute Neo4j query
    # -------------------------

    query_result = execute_cypher(
        cypher
    )


    # -------------------------
    # Step 4
    # Result -> Natural language
    # -------------------------

    answer = generate_answer(
        question=question,
        query_result=query_result,
    )


    # -------------------------
    # Optional debugging
    # -------------------------

    if show_cypher:

        print(
            "\nGenerated Cypher:\n"
        )

        print(cypher)


    if show_raw_result:

        print(
            "\nRaw Neo4j Result:\n"
        )

        print(query_result)


    return answer

In [36]:
question = (
    "Which students belong "
    "to each department?"
)

answer = ask_graph(
    question
)

print(
    "\nAnswer:\n"
)

print(answer)


Generated Cypher:

MATCH (d:Department {graph_name:'university_graph'})-[:HAS_STUDENT]->(s:Student {graph_name:'university_graph'})
RETURN d.identifier AS department, s.identifier AS student
ORDER BY d.identifier, s.identifier

Answer:

**Department: dept_cs**  
- student_001  
- student_002
